## Demonstration Notebook for using the dataloaders

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from magsr import ROOT_FOLDER
from magsr.datasets import build_ksa_aligned_datasets, build_wa_datasets

### Build a Dataset
For Western Australia, the test region is already reserved, however we choose a split and seed for the train/validation sets from the non-test region. See [Study Area Overview](../figures/wa_study_area_overview.png) for a visual overview of the study area and splits.



In [ ]:

# Any field on `WAConfig` (see configs/datasets.yaml:wa) can be overridden as a
# kwarg; unspecified keys fall back to the YAML. `build_wa_datasets()` with no
# args is the happy path — it loads everything from the YAML.
splits = build_wa_datasets(
    val_fraction=0.1,
    seed=44,
)

ds_wa_train = splits["train"]
ds_wa_val = splits["val"]
ds_wa_test = splits["test"]

print(
    f"Datasets ready: train={len(ds_wa_train)}, "
    f"val={len(ds_wa_val)}, test={len(ds_wa_test)}"
)

# Pull a training sample
sample = ds_wa_train[0]
hr_patch = sample["hr"]["MAG"]
lr_patch = sample["lr"]["MAG"]
print(f"HR patch shape: {hr_patch.shape}, LR patch shape: {lr_patch.shape}")

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.title("LR MAG")
plt.imshow(lr_patch, cmap="gray")
plt.subplot(1, 2, 2)
plt.title("HR MAG")
plt.imshow(hr_patch, cmap="gray")
plt.show()

### KSA Shield (aligned) dataset

The three original SGS RGP survey blocks, co-registered onto one **EPSG:32637**
grid so HR (60 m), LR (180 m), and DEM (30 m) share exact integer pixel ratios and
each patch is a plain windowed read — no per-patch warping. Build it from raw with
`scripts/build_ksa_dataset/` (see that folder's READMEs).

**HR products** (`snapped_cubicspline_MAG_<key>.tif`). `AMF` is the IGRF-corrected
aeromagnetic **anomaly** (not raw TMI); the `AMF_*` names are Geosoft-style
derivatives of it.

| Product      | Meaning |
|--------------|---|
| `AMF`        | **Aeromagnetic anomaly field** — total-field scalar after IGRF removal (nT). |
| `AMF_RTP`    | **Reduced To Pole** — as if measured at the magnetic pole, anomalies over their sources. |
| `AMF_RTP1VD` | **1st vertical derivative of RTP** (∂M/∂z) — enhances shallow/short-wavelength features. |
| `AMF_RTP2VD` | **2nd vertical derivative of RTP** — zero crossings approximate source edges. |
| `AMF_RTPHG`  | **Horizontal gradient magnitude of RTP** — peaks over vertical source contacts. |
| `AMF_RTPTILT`| **Tilt derivative of RTP** — amplitude-normalized edge detector. |
| `AMF_ANS`    | **Analytic signal** of AMF — peaks over sources independent of magnetization direction. |
| `HGLAT`      | N–S component of the horizontal gradient. |
| `HGLONG`     | E–W component of the horizontal gradient. |

The 30 m **DEM** is a separate top-level channel (`snapped_cubicspline_dem30m.tif`),
not an HR product, because it lives on its own grid.

**LR products** at 180 m, using the same canonical keys. `TMI` (raw total field,
before IGRF removal) is *not* the same product as HR `AMF`.

| LR key | Meaning |
|--------|---|
| `TMI`  | Total magnetic intensity at 180 m. |
| `RTP`  | Reduced-To-Pole at 180 m. |
| `ANS`  | Analytic signal at 180 m. |
| `1VD`  | First vertical derivative at 180 m. |

#### Loading aligned patches

HR (60 m), LR (180 m, already interpolated), and DEM (30 m) all share one
**EPSG:32637** grid with exact integer pixel ratios:

> `1 LR px = 3×3 HR px = 6×6 DEM px`

Each patch is a plain windowed read out of three co-registered rasters. `patch_px`
(and `stride_px`) must be a multiple of `lr_scale` (3) so HR origins map to integer
LR offsets.

**Sample layout**: `{'hr': {...}, 'lr': {...}, 'dem': tensor, 'meta': {...}}` — DEM
is a top-level key (it lives on its own 30 m grid); `meta` carries the patch
center's `lat`/`lon`. This is native 44→132 (×3) super-resolution.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from magsr.datasets.ksa_shield_aligned import KSAAlignedConfig
from pathlib import Path
import rasterio


def load_raster(path: str) -> np.ndarray:
    p = Path(path)
    if p.suffix in ('.tif', '.tiff'):
        with rasterio.open(p) as src:
            arr = src.read(1).astype(np.float32)
            nd = src.nodata
        if nd is not None and not np.isnan(nd):
            arr[arr == nd] = np.nan
    elif p.suffix == '.npy':
        arr = np.load(p).astype(np.float32)
    else:
        raise ValueError(f'Unsupported format: {p.suffix}')
    return arr

def block_mean_scale_down(hr: np.ndarray, scale: int, lr_shape: tuple[int, int],
                   chunk_rows: int = 512) -> np.ndarray:
    """NaN-aware 3x3 block mean of `hr` onto an LR grid, streamed in row chunks."""
    H, W = lr_shape
    out = np.empty((H, W), dtype=np.float32)
    for r0 in range(0, H, chunk_rows):
        r1 = min(r0 + chunk_rows, H)
        block = hr[r0 * scale: r1 * scale, : W * scale].reshape(r1 - r0, scale, W, scale)
        valid = np.isfinite(block)
        s = np.where(valid, block, 0.0).sum(axis=(1, 3))
        n = valid.sum(axis=(1, 3))
        out[r0:r1] = np.where(n > 0, s / np.maximum(n, 1), np.nan)
    return out

# Load the LR and HR rasters across KSA
cfg = KSAAlignedConfig.default()
hr = load_raster(cfg.hr_product_path("AMF_RTP"))   # 60 m
lr = load_raster(cfg.lr_product_path("RTP"))       # 180 m

# Downsample the HR raster to pixel-match the LR grid
hr_lr = block_mean_scale_down(hr, cfg.lr_scale, lr.shape)
del hr  # free the 1.5 GB float32 array — we don't need it anymore

# Compute the residual (LR − HR on the LR grid)
residual = lr - hr_lr

def nan_gaussian(a, sigma, min_weight=0.05):
    valid = np.isfinite(a).astype(np.float32)
    num = gaussian_filter(np.where(valid > 0, a, 0.0), sigma=sigma)
    den = gaussian_filter(valid, sigma=sigma)
    return np.where(den > min_weight, num / np.maximum(den, 1e-9), np.nan)

smooth = nan_gaussian(residual, sigma=5)
clim = float(np.nanpercentile(np.abs(residual), 98))

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(hr_lr, cmap="gray"); axes[0].set_title("HR AMF_RTP (avg → LR grid)")
axes[1].imshow(lr, cmap="gray"); axes[1].set_title("LR RTP")
axes[2].imshow(residual, cmap="RdBu_r", vmin=-clim, vmax=clim)
axes[2].set_title(f"LR − HR  [μ={np.nanmean(residual):+.1f}, σ={np.nanstd(residual):.1f} nT]")
axes[3].imshow(smooth, cmap="RdBu_r", vmin=-clim, vmax=clim)
axes[3].set_title("Smoothed residual")
for ax in axes: ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

plt.figure(figsize=(7, 3))
plt.hist(residual[np.isfinite(residual)].ravel(), bins=200)
plt.axvline(np.nanmean(residual), color="r", lw=1, label=f"μ={np.nanmean(residual):+.1f}")
plt.xlabel("LR − HR (nT)"); plt.ylabel("LR pixels"); plt.legend(); plt.show()


## Torch Integration

A simple collate function converts the datset dict outputs to a dictionary of batch/channel torch tensors.

In [ ]:
from torch.utils.data import DataLoader
from magsr.datasets import pool_collate, worker_init_fn

batch_size = 64

splits = build_ksa_aligned_datasets(
    index_dir=ROOT_FOLDER / "data/processed/ksa_aligned/patch_indices_cellgrid8_fold3",
    load_dem=True,  # default config has load_dem=False; enable it for the DEM channel
)
loader_ksa_al_train, loader_ksa_al_val, loader_ksa_al_test = [
    DataLoader(
        splits[k],
        collate_fn=pool_collate,
        batch_size=batch_size,
        num_workers=4,
        worker_init_fn=worker_init_fn,
    )
    for k in ["train", "val", "test"]
]

batch = next(iter(loader_ksa_al_train))
hr, lr, dem, meta = [batch[k] for k in ["hr", "lr", "dem", "meta"]]
print(f"Batch shapes: hr={hr.shape}, lr={lr.shape}, dem={dem.shape}, meta={len(meta)}")

# Show batch with einops
from einops import rearrange

# (1, 4*132, 4*132) -> (528, 528)
lr_grid = rearrange(lr, '(b1 b2) c h w -> c (b1 h) (b2 w)', b1=8, b2=8)
hr_grid = rearrange(hr, '(b1 b2) c h w -> c (b1 h) (b2 w)', b1=8, b2=8)
dem_grid = rearrange(dem, '(b1 b2) c h w -> c (b1 h) (b2 w)', b1=8, b2=8)

# Plotting
fig, ax = plt.subplots(1, 3, figsize=(15, 5))

ax[0].imshow(lr_grid[0], cmap='gray')
ax[0].set_title("LR RTP")

ax[1].imshow(hr_grid[0], cmap='gray')
ax[1].set_title("HR AMF_RTP")

ax[2].imshow(dem_grid[0], cmap='terrain')
ax[2].set_title("DEM (30 m terrain)")

plt.show()
